In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# load model
model = load_model("QuickdrawEH.h5")

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 15, 6)]           0         
                                                                 
 gru (GRU)                   (None, 120)               46080     
                                                                 
 dense_0 (Dense)             (None, 50)                6050      
                                                                 
 dense_1 (Dense)             (None, 10)                510       
                                                                 
 output_softmax (Dense)      (None, 3)                 33        
                                                                 
Total params: 52673 (205.75 KB)
Trainable params: 52673 (205.75 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
import numpy as np

# bin = np.binary_repr(num, width=width)

# converts decimal number as shows up in modelsim to actual decimal number
# since verilog implementation uses fixed point and python does not
def fixed_to_dec(num, nfrac):
    return num / (2 ** nfrac)

# vice versa
def dec_to_fixed(num, nfrac):
    return num * (2 ** nfrac)

In [ ]:
import tensorflow as tf

# grab GRU layer from overall model
gru = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("gru").output
)

# constant -0.0009765625 input
x = np.full((1, 100, 3), -0.0009765625, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for -1 input:", gru_output)

# constant 0 input
x = np.full((1, 100, 3), 0, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for 0 input:", gru_output)

# randomly generated input
# TODO: change this to the right dimensions
x = np.array([[
  [ 1.23828125, -0.5546875 ,  3.3203125 ,  4.125     , -1.66796875,  2.75       ],
  [ -4.03125   ,  1.99609375, -2.5       ,  3.11132812,  0.875     , -1.21875   ],
  [  1.0625    , -2.4453125 ,  0.9375    , -4.99902344,  4.5       ,  0.78125   ],
  [ -2.125     ,  4.52      , -2.75      ,  0.03125   , -1.875     ,  3.33300781],
  [  4.0       , -4.59      ,  0.25      , -3.125     ,  4.0625    , -0.9375    ],
  [  2.77734375, -1.66601562,  3.88867188, -1.12304688,  3.2109375 , -1.875     ],
  [  1.5       , -4.25      ,  2.8125    ,  4.125     , -2.75      ,  1.1015625 ],
  [ -2.4375    ,  4.0       , -0.5       ,  0.875     , -4.03125   ,  0.5       ],
  [  0.40039062, -0.30078125,  2.20019531, -1.10058594,  0.05078125, -2.02539062],
  [  0.75      , -0.875     ,  2.9375    , -4.0625    ,  3.75      , -1.5       ],
  [  3.875     , -1.125     ,  4.5       , -4.75      ,  2.625     , -4.3125    ],
  [  0.0       , -3.75      ,  1.125     , -1.499     ,  2.25      , -3.75      ],
  [  4.44433594, -0.55566406,  1.66601562, -2.77734375,  3.88867188, -4.99902344],
  [  0.125     , -1.25      ,  2.375     , -3.5       ,  4.625     , -0.75      ],
  [  1.9375    , -2.8125    ,  3.6875    , -4.5625    ,  0.4375    , -1.3125    ]
]], dtype=np.float32)

gru_output = gru.predict(x)
print("GRU output for random input:", gru_output)

1/1 [==============================] - 0s 302ms/step
GRU output shape: (1, 120)
GRU output for -1 input: [[-0.56890726 -0.09567672 -0.34913725 -0.99494064  0.3613983   0.19687709
  -0.05769272 -0.14897995  0.1877181  -0.03550649 -0.24532282  0.10750702
  -0.11598816 -0.18015094 -0.16135089 -0.85767335 -0.27906138  0.7703383
  -0.5340335   0.05125267 -0.25609422 -0.25098908  0.4315106  -0.37832123
  -0.20760763  0.7366304  -0.07465351  0.7162866   0.43648556  0.05374935
  -0.70293635  0.7977078   0.28382617  0.27658048 -0.54644835  0.12203787
   0.01980045  0.13664083  0.20438394  0.8159071  -0.27387103 -0.2850782
  -0.6261986  -0.11498982 -0.68637383  0.04770102 -0.1362696   0.09853638
  -0.81951106  0.65336573 -0.50525874 -0.1107319  -0.17950605  0.00516415
  -0.18323974 -0.647145   -0.05634918  0.03898488  0.03517921 -0.11741739
  -0.5549089   0.50507605 -0.02983442 -0.88004863 -0.06596029  0.03512058
   0.6822395  -0.19708467 -0.32967156 -0.45299914  0.52376145  0.25188696
  -0.6143

In [ ]:
# manually run through GRU computation

import numpy as np

# =======================
# set config
# =======================
TIMESTEPS = 100
INPUT_SIZE = 3
HIDDEN_SIZE = 128

# =======================
# helper functions
# =======================
def load_vector(filename):
    return np.loadtxt(filename)

def load_matrix(filename, rows, cols):
    data = np.loadtxt(filename)
    return data.reshape(rows, cols)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =======================
# get GRU weights
# =======================

gru = model.layers[1]

kernel, recurrent_kernel, bias = gru.get_weights()

# split kernels
W_z, W_r, W_h = np.split(kernel, 3, axis=1)
U_z, U_r, U_h = np.split(recurrent_kernel, 3, axis=1)

# split bias into input and recurrent
b_input = bias[0]   # (360,)
b_rec   = bias[1]   # (360,)

# split per gate (update, reset, candidate hidden state)
b_z, b_r, b_h = np.split(b_input, 3)
b_z_rec, b_r_rec, b_h_rec = np.split(b_rec, 3)

# =======================
# set input
# =======================
x = np.full((1, 100, 3), 0, dtype=np.float32)

# =======================
# initialize hidden state
# =======================
h = np.zeros(HIDDEN_SIZE) # more zeroes

# trace storage
z_trace = []
z_raw_trace = []
r_trace = []
r_raw_trace = []
h_tilde_trace = []
h_tilde_raw_trace = []
h_trace = []

# =======================
# GRU calculation
# =======================
for t in range(TIMESTEPS):
    x_t = x[t]

    # gates
    z_raw = x_t @ W_z + h @ U_z + b_z + b_z_rec
    r_raw = x_t @ W_r + h @ U_r + b_r + b_r_rec
    z = sigmoid(z_raw)
    r = sigmoid(r_raw)

    # candidate (reset_after=True)
    h_tilde_raw = x_t @ W_h + b_h + r * (h @ U_h + b_h_rec)
    h_tilde = np.tanh(h_tilde_raw)

    # hidden state update
    h = (1 - z) * h_tilde + z * h

    # save traces
    z_trace.append(z)
    z_raw_trace.append(z_raw)
    r_trace.append(r)
    r_raw_trace.append(r_raw)
    h_tilde_trace.append(h_tilde)
    h_tilde_raw_trace.append(h_tilde_raw)
    h_trace.append(h)

# convert to arrays
z_raw_trace = np.array(z_raw_trace)
z_trace = np.array(z_trace)
r_raw_trace = np.array(r_raw_trace)
r_trace = np.array(r_trace)
h_tilde_trace = np.array(h_tilde_trace)
h_tilde_raw_trace = np.array(h_tilde_raw_trace)
h_trace = np.array(h_trace)

# =======================
# outputs
# =======================
print("final hidden state (first 10 units):")
print(h[:10])

print("\nhidden state evolution (first unit):")
print(h_trace[:, 0])

final hidden state (first 10 units):
[ 0.99571936 -0.73386277  0.33621151  0.79067215 -0.98686783  0.96695117
 -0.67945755  0.42269132 -0.97166419  0.28731682]

hidden state evolution (first unit):
[-0.0011294  -0.01602258  0.94032347  0.74517529  0.99996411  0.99998205
 -0.94073619 -0.98490441  0.87385254  0.99124663  0.99999904 -0.36259174
  0.9992689   0.61226292  0.99571936]


In [11]:
# print detailed evolution of what signals should look like in modelsim

print("evolution in modelsim (by timestep):\n")

num_steps = min(15, r_trace.shape[1])
nfrac = 16

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = dec_to_fixed(r_trace[t, :5], nfrac)
    z_fix = dec_to_fixed(z_trace[t, :5], nfrac)
    h_tilde_fix = dec_to_fixed(h_tilde_trace[t, :5], nfrac)
    h_fix = dec_to_fixed(h_trace[t, :5], nfrac)
    
    r_raw = dec_to_fixed(r_raw_trace[t, :5], nfrac)
    z_raw = dec_to_fixed(z_raw_trace[t, :5], nfrac)
    h_tilde_raw = dec_to_fixed(h_tilde_raw_trace[t, :5], nfrac)
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution in modelsim (by timestep):

t = 0
  r_raw   : [    388.65789795   -3616.93798828   47115.49768066 -149189.87304688
  138306.43701172]
  r       : [32865.16418971 31863.99495386 44064.46312041  6100.85494298
 58452.13624379]
  z_raw   : [-341708.94140625  -29683.21191406   17782.24414062  -67722.96960449
  -50778.74121094]
  z       : [  354.55569886 25471.50943743 37186.4857404  17198.6900028
 20672.47354986]
  h_tilde_raw : [-7.44192877e+01 -1.57871899e+05 -3.03227035e+04  1.03580009e+05
  3.18874984e+05]
  h_tilde : [   -74.41925572 -64484.9104721  -28329.39460829  60206.66497373
  65528.21537646]
  h       : [   -74.0166408  -39421.92210446 -12254.7085024   44406.55866596
  44858.19738422]

t = 1
  r_raw   : [409119.44258751 194979.68065839  23707.73162155 -31986.40529777
 -93714.08099439]
  r       : [65408.80201114 62353.50322421 38631.13252087 24926.449392
 12655.28916697]
  z_raw   : [ 274641.68111329 -314313.091094     30047.03899536   28959.61364817
   94835.32681961

In [ ]:
# print detailed evolution of what signals should look like

print("evolution (by timestep):\n")

num_steps = min(15, r_trace.shape[1])
nfrac = 16

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = r_trace[t, :5]
    z_fix = z_trace[t, :5]
    h_tilde_fix = h_tilde_trace[t, :5]
    h_fix = h_trace[t, :5]
    
    r_raw = r_raw_trace[t, :5]
    z_raw = z_raw_trace[t, :5]
    h_tilde_raw = h_tilde_raw_trace[t, :5]
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution (by timestep):

t = 0
  r_raw   : [ 0.00593045 -0.05519009  0.71892544 -2.2764568   2.11038875]
  r       : [0.50148261 0.48620598 0.67237035 0.09309166 0.89190882]
  z_raw   : [-5.21406466 -0.45292987  0.27133551 -1.03337051 -0.7748221 ]
  z       : [0.00541009 0.38866439 0.56742074 0.26243118 0.31543691]
  h_tilde_raw : [-1.13554821e-03 -2.40893401e+00 -4.62687737e-01  1.58050550e+00
  4.86564611e+00]
  h_tilde : [-0.00113555 -0.98396165 -0.43227226  0.9186808   0.99988122]
  h       : [-0.0011294  -0.60153079 -0.18699201  0.67759031  0.68448177]

t = 1
  r_raw   : [ 6.24266728  2.97515382  0.36175128 -0.48807381 -1.42996339]
  r       : [0.99805911 0.95143895 0.5894643  0.38034743 0.19310439]
  z_raw   : [ 4.19069948 -4.79603716  0.45848143  0.44188864  1.44707225]
  z       : [0.98508998 0.00819472 0.61265387 0.60870896 0.80954744]
  h_tilde_raw : [-8.03536958 -3.60945324 -0.0271261  -1.11719587  2.21856165]
  h_tilde : [-0.99999979 -0.99853587 -0.02711944 -0.80659134  0.

In [ ]:
x = np.full((1, 100, 3), 0, dtype=np.float32)

intermediate_output = x
for i, layer in enumerate(model.layers):
    intermediate_output = layer(intermediate_output)
    print(f"Layer {i} ({layer.name}) output: {[intermediate_output.numpy()[:5]]}")